In [2]:
from sqlalchemy import create_engine
import pandas as pd

# Change these
username = "root"
password = "moon98"
host = "localhost"
port = 3306
database = "db2"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

conn = engine.connect()

print("Connected successfully!")

Connected successfully!


In [8]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS test (
            id INT PRIMARY KEY,
            name VARCHAR(50)
        );
    """))

In [7]:
df = pd.read_csv("data/Clean_data.csv")

df.to_sql(
    "cars",
    con=engine,
    if_exists="replace",   # or "append"
    index=False
)

print("Data uploaded!")

Data uploaded!


In [9]:
query = """
SELECT fuel, COUNT(*) AS total
FROM cars
GROUP BY fuel
ORDER BY total DESC;
"""

pd.read_sql(query, engine)

,fuel,total
0,petrol,21719
1,diesel,14587
2,cng,1071
3,lpg,203
4,electric,128


In [14]:
# Q1: Total number of cars

query = """
SELECT COUNT(*) AS total_cars
FROM cars;
"""

pd.read_sql(query, engine)

,total_cars
0,37708


In [15]:
# Cars by fuel type

query = """
SELECT fuel,
       COUNT(*) AS total_cars
FROM cars
GROUP BY fuel
ORDER BY total_cars DESC;
"""

pd.read_sql(query, engine)

,fuel,total_cars
0,petrol,21719
1,diesel,14587
2,cng,1071
3,lpg,203
4,electric,128


In [20]:
# Average selling price by fuel type

query = """
SELECT fuel,
       ROUND(AVG(Listed_price),2) AS Listed_price
FROM cars
GROUP BY fuel
ORDER BY Listed_price DESC;
"""

pd.read_sql(query, engine)

,fuel,Listed_price
0,electric,1484946.88
1,diesel,1030397.59
2,petrol,634490.03
3,cng,457274.39
4,lpg,177267.72


In [23]:
# Top 10 most expensive cars

query = """
SELECT oem,
       model,
       Listed_price
FROM cars
ORDER BY Listed_price DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,oem,model,Listed_price
0,rolls-royce,rolls-royce phantom,95519530
1,ferrari,ferrari sf90 stradale,44425000
2,ferrari,ferrari 488 gtb,32570569
3,lamborghini,lamborghini huracan evo,29999999
4,mercedes-benz,mercedes-benz maybach s-class,28300000
5,lamborghini,lamborghini huracan,22500000
6,aston martin,aston martin vantage,21100000
7,mercedes-benz,mercedes-benz s-class,19768726
8,mercedes-benz,mercedes-benz s-class,19500000
9,mercedes-benz,mercedes-benz s-class,19000000


In [26]:
# Average selling price by transmission

query = """
SELECT transmission,
       COUNT(*) AS total_cars,
       ROUND(AVG(Listed_price),2) AS Listed_price
FROM cars
GROUP BY transmission
ORDER BY Listed_price DESC;
"""

pd.read_sql(query, engine)

,transmission,total_cars,Listed_price
0,automatic,8867,1626210.22
1,manual,28841,523806.11


In [28]:
# Top 10 brands by number of listings

query = """
SELECT oem,
       COUNT(*) AS total_cars
FROM cars
GROUP BY oem
ORDER BY total_cars DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,oem,total_cars
0,maruti,9930
1,hyundai,7399
2,honda,3898
3,mahindra,2132
4,tata,2020
5,toyota,1236
6,ford,1181
7,renault,1177
8,volkswagen,1057
9,skoda,1017


In [30]:
# Average selling price by body type

query = """
SELECT body,
       COUNT(*) AS total_cars,
       ROUND(AVG(Listed_price),2) AS Listed_price
FROM cars
GROUP BY body
ORDER BY Listed_price DESC;
"""

pd.read_sql(query, engine)

,body,total_cars,Listed_price
0,coupe,20,11222829.05
1,hybrids,4,5400000.00
2,convertibles,22,5018364.64
3,wagon,7,2263947.86
4,luxury vehicles,71,2061484.13
5,suv,9789,1289782.80
6,pickup trucks,47,943831.81
7,muv,1810,904885.49
8,sedan,10512,829867.38
9,hatchback,15022,388742.75


In [33]:
# Cars priced above the overall average

query = """
SELECT oem,
       model,
       Listed_price
FROM cars
WHERE Listed_price >
(
    SELECT AVG(Listed_price)
    FROM cars
)
ORDER BY Listed_price DESC;
"""

pd.read_sql(query, engine)

,oem,model,Listed_price
0,rolls-royce,rolls-royce phantom,95519530
1,ferrari,ferrari sf90 stradale,44425000
2,ferrari,ferrari 488 gtb,32570569
3,lamborghini,lamborghini huracan evo,29999999
4,mercedes-benz,mercedes-benz maybach s-class,28300000
...,...,...,...
10979,maruti,maruti swift,784000
10980,kia,kia sonet,784000
10981,maruti,maruti vitara brezza,783915
10982,tata,tata tigor,783239


In [34]:
# Mileage category analysis

query = """
SELECT
CASE
    WHEN km <= 25000 THEN '0-25K'
    WHEN km <= 50000 THEN '25K-50K'
    WHEN km <= 75000 THEN '50K-75K'
    WHEN km <= 100000 THEN '75K-100K'
    ELSE '100K+'
END AS mileage_group,
COUNT(*) AS total_cars,
ROUND(AVG(Listed_price),2) AS Listed_price
FROM cars
GROUP BY mileage_group
ORDER BY Listed_price DESC;
"""

pd.read_sql(query, engine)

,mileage_group,total_cars,Listed_price
0,0-25K,7034,1208532.36
1,25K-50K,9573,869727.20
2,50K-75K,9087,685977.61
3,75K-100K,6552,580079.85
4,100K+,5462,488066.89


In [35]:
# Average selling price by manufacturing year

query = """
SELECT myear,
       COUNT(*) AS total_cars,
       ROUND(AVG(Listed_price),2) AS Listed_price
FROM cars
GROUP BY myear
ORDER BY myear DESC;
"""

pd.read_sql(query, engine)

,myear,total_cars,Listed_price
0,2023,35,3111828.14
1,2022,1278,1582478.92
2,2021,2510,1395415.63
3,2020,2297,1223475.86
4,2019,2951,1113648.58
5,2018,3662,1011579.53
6,2017,3913,918749.54
7,2016,3410,753370.04
8,2015,3457,670004.44
9,2014,3479,546612.99
